## 2. Imports

In [11]:
import os
import shutil
import numpy as np
import soundfile as sf
import librosa
from scipy import signal
from tqdm import tqdm
from pathlib import Path
import io
from pydub import AudioSegment

# Seed for reproducibility
np.random.seed(42)
print("All imports successful.")

All imports successful.


## 3. Define Paths


In [12]:
# ── Edit these two paths ──────────────────────────────────────────────────────
PREPROCESSED_DIR = "../data/preprocessed"  
AUGMENTED_DIR    = "../data/augmented"       
# ─────────────────────────────────────────────────────────────────────────────

PREPROCESSED_DIR = Path(PREPROCESSED_DIR)
AUGMENTED_DIR    = Path(AUGMENTED_DIR)

print(f"Source : {PREPROCESSED_DIR.resolve()}")
print(f"Output : {AUGMENTED_DIR.resolve()}")
print(f"Source exists: {PREPROCESSED_DIR.exists()}")

Source : /Users/Israr/Documents/Personal/Lums/S02 - 600/Project/Deepfake-Audio-Detection-Real-vs.-Synthetically-Cloned-Voices/data/preprocessed
Output : /Users/Israr/Documents/Personal/Lums/S02 - 600/Project/Deepfake-Audio-Detection-Real-vs.-Synthetically-Cloned-Voices/data/augmented
Source exists: True


## 4. Mirror Folder Structure



In [13]:
def mirror_folder_structure(src: Path, dst: Path):
    """Create the same directory tree in dst as exists in src."""
    for dirpath, dirnames, _ in os.walk(src):
        relative = Path(dirpath).relative_to(src)
        target   = dst / relative
        target.mkdir(parents=True, exist_ok=True)
    print(f"Folder structure mirrored: {dst}")

mirror_folder_structure(PREPROCESSED_DIR, AUGMENTED_DIR)

Folder structure mirrored: ../data/augmented


## 5. RawBoost+ Augmentation

RawBoost+ combines three noise types:
| Noise Type | Description |
|---|---|
| **LN** – Linear Convolutive | Filters the signal through a random FIR filter |
| **NN** – Non-linear Convolutive | Introduces harmonic/clipping distortion |
| **ISD** – Impulsive Signal-Dependent | Adds sparse, signal-scaled impulse noise |

In [14]:
# ── RawBoost Helper Functions ─────────────────────────────────────────────────

def normalize(x):
    """Normalize audio to [-1, 1]."""
    peak = np.max(np.abs(x))
    return x / (peak + 1e-8)


def linear_convolutive_noise(x, sr, snr_db=20, filter_order=32):
    """
    LN: Convolve signal with a random FIR filter, then add at target SNR.
    """
    h = np.random.randn(filter_order)
    h /= np.linalg.norm(h) + 1e-8

    noise = signal.fftconvolve(x, h, mode='same')

    signal_power = np.mean(x ** 2)
    noise_power  = np.mean(noise ** 2) + 1e-8
    snr_linear   = 10 ** (snr_db / 10)
    scale        = np.sqrt(signal_power / (snr_linear * noise_power))

    return normalize(x + scale * noise)


def nonlinear_convolutive_noise(x, sr, snr_db=20, filter_order=32):
    """
    NN: Apply soft-clipping non-linearity then convolve with random FIR filter.
    """
    # Soft clipping (non-linear distortion)
    x_nl = np.tanh(3 * x)

    h = np.random.randn(filter_order)
    h /= np.linalg.norm(h) + 1e-8
    noise = signal.fftconvolve(x_nl - x, h, mode='same')  # distortion component only

    signal_power = np.mean(x ** 2)
    noise_power  = np.mean(noise ** 2) + 1e-8
    snr_linear   = 10 ** (snr_db / 10)
    scale        = np.sqrt(signal_power / (snr_linear * noise_power))

    return normalize(x + scale * noise)


def impulsive_signal_dependent_noise(x, sr, snr_db=20, sparsity=0.005):
    """
    ISD: Sparse impulses scaled by the local signal amplitude.
    """
    impulse_mask = np.random.rand(len(x)) < sparsity
    noise        = impulse_mask * x * np.random.randn(len(x))

    signal_power = np.mean(x ** 2)
    noise_power  = np.mean(noise ** 2) + 1e-8
    snr_linear   = 10 ** (snr_db / 10)
    scale        = np.sqrt(signal_power / (snr_linear * noise_power))

    return normalize(x + scale * noise)


def rawboost_plus(x, sr, snr_db=20):
    """
    RawBoost+: sequentially applies LN → NN → ISD.
    """
    x = linear_convolutive_noise(x, sr, snr_db=snr_db)
    x = nonlinear_convolutive_noise(x, sr, snr_db=snr_db + 5)  # slightly less aggressive
    x = impulsive_signal_dependent_noise(x, sr, snr_db=snr_db + 5)
    return x


print("RawBoost+ functions defined.")

RawBoost+ functions defined.


## 6. Codec Augmentation


In [15]:
def codec_augmentation(x, sr, codec='mp3', bitrate='32k'):
    """
    Encode audio to a lossy codec and decode back to simulate compression.

    Parameters
    ----------
    x       : np.ndarray  – audio signal (float32, range [-1,1])
    sr      : int         – sample rate
    codec   : str         – 'mp3' or 'opus'
    bitrate : str         – e.g. '32k', '64k'

    Returns
    -------
    np.ndarray – codec-distorted audio, same length as input
    """
    # Convert float32 → int16 PCM for pydub
    pcm = (x * 32767).astype(np.int16)
    audio_segment = AudioSegment(
        pcm.tobytes(),
        frame_rate=sr,
        sample_width=2,   # 16-bit
        channels=1
    )

    buf = io.BytesIO()

    if codec == 'opus':
        # opus must use ogg as container
        audio_segment.export(buf, format='ogg', codec='libopus', bitrate=bitrate)
        buf.seek(0)
        decoded = AudioSegment.from_file(buf, format='ogg')
    else:
        # mp3
        audio_segment.export(buf, format='mp3', bitrate=bitrate)
        buf.seek(0)
        decoded = AudioSegment.from_file(buf, format='mp3')

    samples = np.array(decoded.get_array_of_samples()).astype(np.float32) / 32767.0

    # Match original length (codec may add/remove frames)
    if len(samples) >= len(x):
        samples = samples[:len(x)]
    else:
        samples = np.pad(samples, (0, len(x) - len(samples)))

    return normalize(samples)


print("Codec augmentation function defined.")

Codec augmentation function defined.


## 7. Combined Augmentation Pipeline

Each file randomly receives one of three augmentation strategies:
- RawBoost+ only
- Codec only
- RawBoost+ **then** Codec (most aggressive)

In [16]:
def augment_audio(x, sr, strategy=None):
    """
    Apply a randomly selected augmentation strategy to a waveform.

    Strategies (equal probability when strategy=None)
    ----------
    'rawboost_plus'  – RawBoost+ only          (works without ffmpeg)
    'codec'          – Codec compression only   (requires ffmpeg)
    'both'           – RawBoost+ then Codec     (requires ffmpeg)
    """
    if strategy is None:
        strategy = np.random.choice(['rawboost_plus', 'codec', 'both'])

    if strategy == 'rawboost_plus':
        return rawboost_plus(x, sr)

    elif strategy == 'codec':
        codec   = np.random.choice(['mp3', 'opus'])
        bitrate = np.random.choice(['32k', '48k', '64k'])
        return codec_augmentation(x, sr, codec=codec, bitrate=bitrate)

    elif strategy == 'both':
        x = rawboost_plus(x, sr)
        codec   = np.random.choice(['mp3', 'opus'])
        bitrate = np.random.choice(['32k', '48k'])
        return codec_augmentation(x, sr, codec=codec, bitrate=bitrate)

    else:
        raise ValueError(f"Unknown strategy: {strategy}")


print("Augmentation pipeline defined.")

Augmentation pipeline defined.


## 8. Run Augmentation on All Files

Walk through every `.flac` file in `Preprocessed/`, augment it, and save to the mirrored path inside `Augmented/`.

In [17]:
def get_all_audio_files(root: Path, extensions=('.flac', '.wav')):
    """Recursively collect all audio files under root."""
    return [p for p in root.rglob('*') if p.suffix.lower() in extensions]


audio_files = [
    p for p in get_all_audio_files(PREPROCESSED_DIR)
    if 'train' in str(p)
]
dev_count  = sum(1 for p in audio_files if 'dev'  in str(p))
eval_count = sum(1 for p in audio_files if 'eval' in str(p))
print(f"Train files : {len(audio_files)}")
print(f"Dev files   : {dev_count}")    # must be 0
print(f"Eval files  : {eval_count}")   # must be 0
print(f"Found {len(audio_files)} audio files to augment.")

# Preview a few
for f in audio_files[:5]:
    print(" ", f.relative_to(PREPROCESSED_DIR))

Train files : 25380
Dev files   : 0
Eval files  : 0
Found 25380 audio files to augment.
  LA/ASVspoof2019_LA_train/flac/LA_T_3653923.flac
  LA/ASVspoof2019_LA_train/flac/LA_T_9203016.flac
  LA/ASVspoof2019_LA_train/flac/LA_T_7555561.flac
  LA/ASVspoof2019_LA_train/flac/LA_T_5726173.flac
  LA/ASVspoof2019_LA_train/flac/LA_T_8788751.flac


In [18]:
all_files = get_all_audio_files(PREPROCESSED_DIR)

skipped = []
success = 0


for src_path in tqdm(all_files, desc="Processing"):
    try:
        relative = src_path.relative_to(PREPROCESSED_DIR)
        dst_path = AUGMENTED_DIR / relative
        dst_path.parent.mkdir(parents=True, exist_ok=True)


        # ── Dev / Eval → just copy, no augmentation ───────────────────────────
        if 'dev' in str(src_path) or 'eval' in str(src_path):
            shutil.copy2(str(src_path), str(dst_path))

        # ── Train → augment then save ─────────────────────────────────────────
        else:
            x, sr = sf.read(str(src_path), dtype='float32')

            if x.ndim > 1:
                x = x.mean(axis=1)
            x_aug    = augment_audio(x, sr)
            

            sf.write(str(dst_path), x_aug, sr, subtype='PCM_16')


        success += 1

    except Exception as e:
        skipped.append((src_path.name, str(e)))

print(f"\n Done! Processed: {success} | Skipped: {len(skipped)}")
if skipped:
    print("Skipped files:")
    for name, err in skipped:
        print(f"  {name}: {err}")

Processing: 100%|██████████| 122299/122299 [32:44<00:00, 62.27it/s]   


 Done! Processed: 122299 | Skipped: 0


## 9. Verify Output Structure

Confirm the `Augmented/` folder mirrors `Preprocessed/` and all files are present.

In [19]:
augmented_files = get_all_audio_files(AUGMENTED_DIR)

print(f"Preprocessed files : {len(audio_files)}")
print(f"Augmented files    : {len(augmented_files)}")
print()

# Show the Augmented folder tree (top 2 levels)
for p in sorted(AUGMENTED_DIR.rglob('*')):
    depth = len(p.relative_to(AUGMENTED_DIR).parts)
    if depth <= 3:
        indent = '  ' * (depth - 1)
        print(f"{indent}{p.name}{'/' if p.is_dir() else ''}")

Preprocessed files : 25380
Augmented files    : 122299

LA/
  ASVspoof2019_LA_dev/
    flac/
  ASVspoof2019_LA_eval/
    flac/
  ASVspoof2019_LA_train/
    flac/


## 10. Quick Audio Sanity Check (Optional)

Listen to an original file vs its augmented version side-by-side.

In [20]:
from IPython.display import Audio, display

# Find the first train file that was successfully augmented
augmented_train = sorted([
    p for p in get_all_audio_files(AUGMENTED_DIR)
    if 'train' in str(p)
])

if not augmented_train:
    print("No augmented train files found.")
    print("Make sure ffmpeg is installed (brew install ffmpeg) then re-run the augmentation cell.")
else:
    sample_dst = augmented_train[0]
    relative   = sample_dst.relative_to(AUGMENTED_DIR)
    sample_src = PREPROCESSED_DIR / relative

    if not sample_src.exists():
        print(f"Source file not found: {sample_src}")
    else:
        x_orig, sr = sf.read(str(sample_src), dtype='float32')
        x_aug,  _  = sf.read(str(sample_dst), dtype='float32')

        print(f"File: {sample_dst.name}")
        print(f"Original length : {len(x_orig)} samples")
        print(f"Augmented length: {len(x_aug)} samples")
        print()
        print("Original:")
        display(Audio(x_orig, rate=sr))
        print("Augmented:")
        display(Audio(x_aug, rate=sr))

File: LA_T_1000137.flac
Original length : 64600 samples
Augmented length: 64600 samples

Original:


Augmented:
